In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import requests
from io import BytesIO

In [ ]:
def find_optimal_n_methods(singular_values, method='energy_threshold', **kwargs):
    """
    Find optimal n using different methods.
    
    Methods:
    - 'energy_threshold': Keep n that preserves X% of energy (default 95%)
    - 'elbow': Find the elbow point using curvature/Knee point detection
    - 'threshold': Keep singular values above threshold * max_singular_value
    - 'mse_target': Find n that achieves target MSE
    - 'compression_target': Find n that achieves target compression ratio
    - 'manual': Use specified n value
    """
    
    if method == 'energy_threshold':
        # Method 1: Energy threshold (preserve X% of variance)
        energy_threshold = kwargs.get('energy_threshold', 0.95)
        squared_sv = singular_values ** 2
        total_energy = np.sum(squared_sv)
        cumulative_energy = np.cumsum(squared_sv)
        energy_ratios = cumulative_energy / total_energy
        optimal_n = np.argmax(energy_ratios >= energy_threshold) + 1
        return optimal_n, energy_ratios
    
    elif method == 'elbow':
        # Method 2: Elbow/Knee point detection using curvature
        # Find the point where the marginal gain drops significantly
        n_sv = len(singular_values)
        
        # Normalize singular values to [0,1] range
        sv_normalized = singular_values / singular_values[0]
        
        # Calculate differences (marginal gain)
        differences = np.diff(sv_normalized)
        
        # Find where the difference drops below threshold
        # Or use the "knee" point detection algorithm
        knee_threshold = kwargs.get('knee_threshold', 0.1)
        
        # Find first point where cumulative explained variance is high
        # but singular value magnitude is low
        for i in range(1, min(n_sv, 100)):
            if sv_normalized[i] < knee_threshold:
                optimal_n = i
                break
        else:
            optimal_n = min(50, n_sv // 10)
        
        # Calculate energy ratios for consistency
        squared_sv = singular_values ** 2
        total_energy = np.sum(squared_sv)
        cumulative_energy = np.cumsum(squared_sv)
        energy_ratios = cumulative_energy / total_energy
        
        return optimal_n, energy_ratios
    
    elif method == 'threshold':
        # Method 3: Keep singular values above threshold * max_singular_value
        threshold_ratio = kwargs.get('threshold_ratio', 0.01)  # 1% of max by default
        threshold_value = singular_values[0] * threshold_ratio
        optimal_n = np.sum(singular_values > threshold_value)
        
        # Calculate energy ratios for consistency
        squared_sv = singular_values ** 2
        total_energy = np.sum(squared_sv)
        cumulative_energy = np.cumsum(squared_sv)
        energy_ratios = cumulative_energy / total_energy
        
        return optimal_n, energy_ratios
    
    elif method == 'mse_target':
        # Method 4: Find n that achieves target MSE (requires image)
        # Note: This method needs to be called with the image data
        target_mse = kwargs.get('target_mse', 100)  # Default MSE target
        image = kwargs.get('image', None)
        
        if image is None:
            # Fallback to energy threshold
            print("Warning: No image provided for MSE target method, using energy threshold")
            return find_optimal_n_methods(singular_values, 'energy_threshold', energy_threshold=0.95)
        
        # Test different n values to find target MSE
        max_test_n = min(len(singular_values), 100)
        for n_test in range(1, max_test_n + 1):
            # Reconstruct with n_test components
            U = kwargs.get('U', None)
            Vt = kwargs.get('Vt', None)
            
            if U is not None and Vt is not None:
                reconstructed = U[:, :n_test] @ np.diag(singular_values[:n_test]) @ Vt[:n_test, :]
                mse = np.mean((image.astype(np.float64) - reconstructed) ** 2)
                
                if mse <= target_mse:
                    optimal_n = n_test
                    break
        else:
            optimal_n = max_test_n
        
        # Calculate energy ratios
        squared_sv = singular_values ** 2
        total_energy = np.sum(squared_sv)
        cumulative_energy = np.cumsum(squared_sv)
        energy_ratios = cumulative_energy / total_energy
        
        return optimal_n, energy_ratios
    
    elif method == 'compression_target':
        # Method 5: Find n that achieves target compression ratio
        target_ratio = kwargs.get('target_ratio', 10)  # Default 10x compression
        h, w = kwargs.get('image_shape', (100, 100))
        
        # Solve for n: (h*w) / (h*n + n + n*w) = target_ratio
        # (h*w) = target_ratio * (h*n + n + n*w)
        # (h*w) = target_ratio * n * (h + 1 + w)
        # n = (h*w) / (target_ratio * (h + w + 1))
        
        if len(kwargs.get('image_shape', ())) == 2:
            h, w = kwargs['image_shape']
            n_calculated = (h * w) / (target_ratio * (h + w + 1))
            optimal_n = max(1, min(len(singular_values), int(np.ceil(n_calculated))))
        else:
            # Fallback to energy threshold
            print("Warning: Invalid image shape for compression target, using energy threshold")
            return find_optimal_n_methods(singular_values, 'energy_threshold', energy_threshold=0.95)
        
        # Calculate energy ratios
        squared_sv = singular_values ** 2
        total_energy = np.sum(squared_sv)
        cumulative_energy = np.cumsum(squared_sv)
        energy_ratios = cumulative_energy / total_energy
        
        return optimal_n, energy_ratios
    
    elif method == 'manual':
        # Method 6: Manual n value
        optimal_n = kwargs.get('n_value', 50)
        
        # Calculate energy ratios
        squared_sv = singular_values ** 2
        total_energy = np.sum(squared_sv)
        cumulative_energy = np.cumsum(squared_sv)
        energy_ratios = cumulative_energy / total_energy
        
        return optimal_n, energy_ratios
    
    else:
        # Default to energy threshold
        print(f"Unknown method '{method}', using energy_threshold")
        return find_optimal_n_methods(singular_values, 'energy_threshold', energy_threshold=0.95)

def load_image(image_source):
    """
    Load image from file path, URL, or numpy array.
    Returns image as numpy array (RGB format).
    """
    if isinstance(image_source, np.ndarray):
        # Already a numpy array
        if len(image_source.shape) == 2:
            # Grayscale, convert to RGB
            return cv2.cvtColor(image_source, cv2.COLOR_GRAY2RGB)
        return image_source
    
    elif isinstance(image_source, str):
        # Check if it's a URL
        if image_source.startswith(('http://', 'https://')):
            response = requests.get(image_source)
            img = Image.open(BytesIO(response.content))
            return np.array(img)
        else:
            # Local file path
            img = cv2.imread(image_source)
            if img is None:
                raise ValueError(f"Cannot load image from {image_source}")
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        raise ValueError("image_source must be file path, URL, or numpy array")

def compress_image_svd(image, n=None, energy_threshold=0.95):
    """
    Compress an image using SVD with automatic n selection.
    
    Args:
        image: Input image (grayscale or color) as numpy array
        n: Number of components to keep (if None, auto-calculate)
        energy_threshold: Target energy preservation (if n is None)
    
    Returns:
        compressed_image: Reconstructed image
        n_used: Number of components used
        compression_ratio: Size reduction factor
    """
    original_dtype = image.dtype
    
    # Handle color images
    if len(image.shape) == 3:
        h, w, c = image.shape
        compressed_channels = []
        n_values = []
        
        for i in range(c):
            channel = image[:, :, i].astype(np.float64)
            U, S, Vt = np.linalg.svd(channel, full_matrices=False)
            
            if n is None:
                current_n, _ = find_optimal_n(S, energy_threshold)
            else:
                current_n = n
            
            n_values.append(current_n)
            
            # Truncate and reconstruct
            U_trunc = U[:, :current_n]
            S_trunc = np.diag(S[:current_n])
            Vt_trunc = Vt[:current_n, :]
            
            reconstructed = U_trunc @ S_trunc @ Vt_trunc
            compressed_channels.append(np.clip(reconstructed, 0, 255))
        
        compressed_image = np.stack(compressed_channels, axis=2).astype(original_dtype)
        n_used = int(np.mean(n_values))  # Average n across channels
        
        # Compression ratio calculation
        original_size = h * w * c
        compressed_size_per_channel = h * n_used + n_used + n_used * w
        compressed_size = compressed_size_per_channel * c
        compression_ratio = original_size / compressed_size
        
    else:
        # Grayscale image
        h, w = image.shape
        image_float = image.astype(np.float64)
        U, S, Vt = np.linalg.svd(image_float, full_matrices=False)
        
        if n is None:
            n_used, energy_ratios = find_optimal_n(S, energy_threshold)
        else:
            n_used = n
            _, energy_ratios = find_optimal_n(S, energy_threshold)
        
        U_trunc = U[:, :n_used]
        S_trunc = np.diag(S[:n_used])
        Vt_trunc = Vt[:n_used, :]
        
        compressed_image = U_trunc @ S_trunc @ Vt_trunc
        compressed_image = np.clip(compressed_image, 0, 255).astype(original_dtype)
        
        original_size = h * w
        compressed_size = h * n_used + n_used + n_used * w
        compression_ratio = original_size / compressed_size
    
    return compressed_image, n_used, compression_ratio

def analyze_and_compress(image_source, n_used=None, method='energy_threshold', 
                         convert_to_grayscale=False, auto_show_plots=True, **kwargs):
    """
    Full analysis and compression pipeline with multiple n selection methods.
    
    Methods:
    - 'energy_threshold': Preserve X% of energy (use energy_threshold=0.95)
    - 'elbow': Find elbow/knee point (use knee_threshold=0.1)
    - 'threshold': Keep values above threshold (use threshold_ratio=0.01)
    - 'mse_target': Target MSE (use target_mse=100)
    - 'compression_target': Target compression ratio (use target_ratio=10)
    - 'manual': Force specific n (use n_value=50)
    
    Args:
        image_source: File path, URL, or numpy array
        n_used: Override n (if provided, ignores method)
        method: Method to find optimal n (if n_used is None)
        convert_to_grayscale: If True, convert output to grayscale
        auto_show_plots: Whether to display diagnostic plots
        **kwargs: Additional arguments for the selected method
    """
    # Load image
    image = load_image(image_source)
    original_image = image.copy()
    
    # Convert input to grayscale if requested
    if convert_to_grayscale and len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        display_image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    else:
        display_image = image
    
    # Convert to grayscale for analysis
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    
    # Perform SVD on grayscale for analysis
    U, S, Vt = np.linalg.svd(gray.astype(np.float64), full_matrices=False)
    
    # Determine n_used if not provided
    if n_used is None:
        # Add U and Vt to kwargs for methods that need them
        if method == 'mse_target':
            kwargs['U'] = U
            kwargs['Vt'] = Vt
            kwargs['image'] = gray
        
        if method == 'compression_target':
            kwargs['image_shape'] = gray.shape
        
        optimal_n, energy_ratios = find_optimal_n_methods(S, method, **kwargs)
        n_used = optimal_n
        energy_retained_analysis = energy_ratios[n_used - 1] * 100 if n_used <= len(energy_ratios) else 100
    else:
        _, energy_ratios = find_optimal_n_methods(S, 'energy_threshold', energy_threshold=0.95)
        energy_retained_analysis = energy_ratios[min(n_used, len(energy_ratios)) - 1] * 100
    
    if auto_show_plots:
        # Show comparison of different methods if n_used is auto-detected
        if n_used is None:
            fig_method, axes_method = plt.subplots(2, 3, figsize=(15, 8))
            methods_to_test = ['energy_threshold', 'elbow', 'threshold', 'compression_target']
            
            for idx, test_method in enumerate(methods_to_test):
                row = idx // 3
                col = idx % 3
                
                if test_method == 'energy_threshold':
                    test_n, _ = find_optimal_n_methods(S, test_method, energy_threshold=kwargs.get('energy_threshold', 0.95))
                    axes_method[row, col].set_title(f'Energy Threshold\nn={test_n}')
                elif test_method == 'elbow':
                    test_n, _ = find_optimal_n_methods(S, test_method, knee_threshold=kwargs.get('knee_threshold', 0.1))
                    axes_method[row, col].set_title(f'Elbow Method\nn={test_n}')
                elif test_method == 'threshold':
                    test_n, _ = find_optimal_n_methods(S, test_method, threshold_ratio=kwargs.get('threshold_ratio', 0.01))
                    axes_method[row, col].set_title(f'Threshold Method\nn={test_n}')
                elif test_method == 'compression_target':
                    test_n, _ = find_optimal_n_methods(S, test_method, image_shape=gray.shape, target_ratio=kwargs.get('target_ratio', 10))
                    axes_method[row, col].set_title(f'Compression Target\nn={test_n}')
                
                # Show reconstruction for this method
                compressed_test, _, _ = compress_image_svd(original_image, n=test_n)
                if len(compressed_test.shape) == 3:
                    axes_method[row, col].imshow(compressed_test)
                else:
                    axes_method[row, col].imshow(compressed_test, cmap='gray')
                axes_method[row, col].axis('off')
            
            # Hide unused subplot
            axes_method[1, 2].axis('off')
            plt.suptitle('Comparison of Different Methods for Finding Optimal n', fontsize=14)
            plt.tight_layout()
            plt.show()
        
        # Main visualization
        test_ns = [1, 5, 10, 20, 50, n_used]
        test_ns = sorted(list(set(test_ns)))[:6]
        n_plots = len(test_ns)
        fig, axes = plt.subplots(2, n_plots, figsize=(4*n_plots, 8))
        
        if n_plots == 1:
            axes = axes.reshape(2, 1)
        
        # Show reconstructions
        for idx, test_n in enumerate(test_ns):
            if convert_to_grayscale:
                compressed, _, _ = compress_image_svd(image, n=test_n)
                if len(compressed.shape) == 2:
                    compressed_display = cv2.cvtColor(compressed, cv2.COLOR_GRAY2RGB)
                else:
                    compressed_display = compressed
            else:
                compressed_display, _, _ = compress_image_svd(original_image, n=test_n)
            
            axes[0, idx].imshow(compressed_display)
            
            if len(original_image.shape) == 3 and not convert_to_grayscale:
                h, w, c = original_image.shape
                compressed_size = (h * test_n + test_n + test_n * w) * c
                original_size = h * w * c
                ratio = original_size / compressed_size
            else:
                h, w = gray.shape
                compressed_size = h * test_n + test_n + test_n * w
                original_size = h * w
                ratio = original_size / compressed_size
            
            axes[0, idx].set_title(f'n={test_n}\n{ratio:.1f}x smaller', fontsize=10)
            axes[0, idx].axis('off')
        
        # Plot metrics
        axes[1, 0].plot(energy_ratios * 100, linewidth=2)
        axes[1, 0].set_title('Cumulative Energy', fontsize=10)
        axes[1, 0].set_xlabel('n', fontsize=8)
        axes[1, 0].set_ylabel('Energy (%)', fontsize=8)
        axes[1, 0].axvline(x=n_used, color='r', linestyle='--', label=f'n={n_used}')
        axes[1, 0].legend(fontsize=8)
        axes[1, 0].grid(True, alpha=0.3)
        
        axes[1, 1].semilogy(S, linewidth=2)
        axes[1, 1].set_title('Singular Values (log)', fontsize=10)
        axes[1, 1].set_xlabel('Index', fontsize=8)
        axes[1, 1].set_ylabel('log(σ)', fontsize=8)
        axes[1, 1].axvline(x=n_used, color='r', linestyle='--')
        axes[1, 1].grid(True, alpha=0.3)
        
        # Error vs n
        max_n = min(100, len(S))
        n_range = range(1, max_n + 1)
        errors = []
        for test_n in n_range:
            if convert_to_grayscale:
                compressed, _, _ = compress_image_svd(image, n=test_n)
                if len(compressed.shape) == 3:
                    compressed_gray = cv2.cvtColor(compressed, cv2.COLOR_RGB2GRAY)
                else:
                    compressed_gray = compressed
            else:
                compressed, _, _ = compress_image_svd(original_image, n=test_n)
                if len(compressed.shape) == 3:
                    compressed_gray = cv2.cvtColor(compressed, cv2.COLOR_RGB2GRAY)
                else:
                    compressed_gray = compressed
            
            mse = np.mean((gray.astype(np.float64) - compressed_gray.astype(np.float64))**2)
            errors.append(mse)
        
        axes[1, 2].plot(n_range, errors, linewidth=2)
        axes[1, 2].set_title('MSE vs n', fontsize=10)
        axes[1, 2].set_xlabel('n', fontsize=8)
        axes[1, 2].set_ylabel('MSE', fontsize=8)
        axes[1, 2].axvline(x=n_used, color='r', linestyle='--')
        axes[1, 2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # Final compression
    if convert_to_grayscale:
        compressed_image, final_n, compression_ratio = compress_image_svd(image, n=n_used)
        if len(compressed_image.shape) == 3:
            compressed_image = cv2.cvtColor(compressed_image, cv2.COLOR_RGB2GRAY)
    else:
        compressed_image, final_n, compression_ratio = compress_image_svd(original_image, n=n_used)
    
    # Print results
    if len(original_image.shape) == 3 and not convert_to_grayscale:
        h, w, c = original_image.shape
        original_pixels = h * w * c
        compressed_pixels = (h * final_n + final_n + final_n * w) * c
    else:
        h, w = gray.shape
        original_pixels = h * w
        compressed_pixels = h * final_n + final_n + final_n * w
    
    final_compression_ratio = original_pixels / compressed_pixels
    
    print(f"\n{'='*60}")
    print(f"COMPRESSION RESULTS")
    print(f"{'='*60}")
    print(f"Method used: {method if method else 'manual'}")
    print(f"Image: {original_image.shape[1]}x{original_image.shape[0]}")
    print(f"Mode: {'Grayscale' if convert_to_grayscale else 'Color'}")
    print(f"\nn = {final_n}")
    print(f"Energy retained: {energy_retained_analysis:.1f}%")
    print(f"Compression ratio: {final_compression_ratio:.1f}x")
    print(f"Original pixels: {original_pixels:,}")
    print(f"Compressed storage: {compressed_pixels:,} numbers")
    print(f"{'='*60}\n")
    
    metrics = {
        'n_used': final_n,
        'compression_ratio': final_compression_ratio,
        'energy_retained': energy_retained_analysis,
        'method': method,
        'is_grayscale': convert_to_grayscale
    }
    
    return compressed_image, metrics


In [ ]:
if __name__ == "__main__": 
    # Example 1: Energy Threshold Method (default, 95%)
    print("\n1. ENERGY THRESHOLD METHOD (99%):")
    compressed, metrics = analyze_and_compress(
        image_source='img/2026-06-06-01.jpeg',
        method='energy_threshold',
        energy_threshold=0.99,
        convert_to_grayscale=False,
        auto_show_plots=True
    )
    
    # Example 2: Elbow Method
    print("\n2. ELBOW METHOD:")
    compressed, metrics = analyze_and_compress(
        image_source='img/2026-06-06-01.jpeg',
        method='elbow',
        knee_threshold=0.1,
        convert_to_grayscale=False,
        auto_show_plots=True
    )
    
    # Example 3: Threshold Method (keep 2% of max)
    print("\n3. THRESHOLD METHOD (2% of max):")
    compressed, metrics = analyze_and_compress(
        image_source='img/2026-06-06-01.jpeg',
        method='threshold',
        threshold_ratio=0.02,
        convert_to_grayscale=False,
        auto_show_plots=True
    )
    
    # Example 4: Compression Target Method (target 20x compression)
    print("\n4. COMPRESSION TARGET METHOD (20x):")
    compressed, metrics = analyze_and_compress(
        image_source='img/2026-06-06-01.jpeg',
        method='compression_target',
        target_ratio=20,
        convert_to_grayscale=True,
        auto_show_plots=True
    )
    
    # Example 5: Manual n
    print("\n5. MANUAL METHOD (n=25):")
    compressed, metrics = analyze_and_compress(
        image_source='img/2026-06-06-01.jpeg',
        n_used=25,
        convert_to_grayscale=False,
        auto_show_plots=True
    )    
    